# A/B 实验上线前设计

# 1. 实验背景

前序分析发现：2015年10月7日后，来源4占新增用户的80.83%，来源4 D30留存率为2.37%，首周持续使用深度偏低。来源4中首7天仅活跃0–1天的用户占51.49%，该层D30仅0.45%。

这些历史结果只能说明首周活跃深度与D30留存相关，不能证明通过干预提升D2–D7活跃会因果性提高D30。因此提出待验证假设：**针对来源4新用户加强D2–D7持续使用引导，可能提高D30留存。**

# 2. 实验对象

- 实验对象：完成业务映射确认后的来源4新用户（confirmed Source 4 new users）。
- 随机化单位：msno。

同一用户可能在多天重复访问。如果按session分组，同一用户可能同时进入Control和Treatment，产生跨组污染。按用户进行稳定分组可以保证同一msno始终处于同一实验组。

# 3. Control / Treatment

**Control：**现有首周用户体验。

**Treatment：**针对D2–D7持续使用的增强策略，例如：

- 个性化首周内容推荐；
- D2 / D4召回提醒；
- 根据首次播放行为提供后续推荐。

这些是未来待实验验证的产品或运营方案，不是历史KKBox数据中已经发生的真实Treatment。

# 4. 随机分流方案

未来上线时使用稳定Hash进行50/50用户级分流：

    stable_hash(msno) % 100 < 50  -> Control
    otherwise                    -> Treatment

稳定Hash应由实验平台统一实现，并固定实验ID或盐值，避免同一用户在不同时间改变组别。本Notebook只展示未来assignment逻辑，不对历史用户进行伪实验分组或比较。

Assignment本身不等于A/B实验；真实实验还必须记录Treatment exposure和post-treatment outcome。

In [1]:
import hashlib

def future_experiment_group(msno: str, experiment_id: str) -> str:
    """未来实验平台的稳定50/50分流示意；本Notebook不对历史用户执行该函数。"""
    key = f"{experiment_id}:{msno}".encode("utf-8")
    bucket = int(hashlib.sha256(key).hexdigest(), 16) % 100
    return "Control" if bucket < 50 else "Treatment"

# 5. 实验指标

**Primary Metric**

- D30 Retention

**Secondary Metrics**

- D7 Retention
- first_7d_active_days
- first_7d_num_100

**Guardrail Metrics**

- 30-day paying rate
- 30-day revenue / subscription health

D30是主指标，因为最终要判断首周持续使用干预能否改善较长期的新用户留存。D7和首周使用深度帮助解释干预路径，付费与订阅健康指标用于避免只提升活跃却损害商业结果。

# 6. Baseline

从现有processed用户级底表重新读取并验证来源4在2015-10-07至2015-10-31注册用户的用户数、D30留存和每日新增量。日均符合条件用户数仅作为实验招募速度的历史planning proxy，不代表未来每天一定保持相同流量。

In [2]:
from pathlib import Path
import math
import pandas as pd
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_PATH = PROJECT_ROOT / "data/processed/new_user_anomaly_base.csv"
OUTPUT_CSV = PROJECT_ROOT / "outputs/ab_test_sample_size_scenarios.csv"

usecols = ["msno", "registration_date", "registered_via", "first_7d_active_days",
           "first_7d_num_100", "d7_retained", "d30_retained"]
source4_parts = []
for chunk in pd.read_csv(BASE_PATH, usecols=usecols, parse_dates=["registration_date"], chunksize=200_000):
    mask = chunk["registered_via"].eq(4) & chunk["registration_date"].between("2015-10-07", "2015-10-31")
    source4_parts.append(chunk.loc[mask].copy())

source4 = pd.concat(source4_parts, ignore_index=True)
date_index = pd.date_range("2015-10-07", "2015-10-31", freq="D")
daily_users = (source4.groupby("registration_date")["msno"].nunique()
               .reindex(date_index, fill_value=0)
               .rename("daily_new_users"))

baseline_users = source4["msno"].nunique()
baseline_d30 = source4["d30_retained"].mean()
historical_avg_daily_eligible_users = daily_users.mean()

baseline_summary = pd.DataFrame({
    "metric": ["Source 4 users", "D30 Retention", "date coverage", "average daily eligible users"],
    "value": [f"{baseline_users:,}", f"{baseline_d30:.2%}", f"{len(date_index)} days", f"{historical_avg_daily_eligible_users:,.2f}"]
})
baseline_summary

,metric,value
0,Source 4 users,"172,259"
1,D30 Retention,2.37%
2,date coverage,25 days
3,average daily eligible users,"6,890.36"


In [3]:
daily_users.rename_axis("registration_date").reset_index()

,registration_date,daily_new_users
0,2015-10-07,5189
1,2015-10-08,6189
2,2015-10-09,9498
3,2015-10-10,9056
4,2015-10-11,8493
5,2015-10-12,5904
6,2015-10-13,6414
7,2015-10-14,7260
8,2015-10-15,7134
9,2015-10-16,7098


# 7. MDE情景设计

统一设定：alpha = 0.05、power = 0.80、1:1分流、双侧检验。基于实际读取的D30 baseline设计三档absolute MDE：

- Scenario A：+0.3 percentage point，约2.37% → 2.67%；
- Scenario B：+0.5 percentage point，约2.37% → 2.87%；
- Scenario C：+1.0 percentage point，约2.37% → 3.37%。

relative uplift只用于帮助理解情景大小，不是实验结果。

# 8. 样本量计算

使用statsmodels的proportion_effectsize和NormalIndPower.solve_power进行two-sample proportion test样本量计算，并对每组样本量向上取整。

In [4]:
alpha = 0.05
power = 0.80
mde_scenarios = {"Scenario A": 0.003, "Scenario B": 0.005, "Scenario C": 0.010}
power_analysis = NormalIndPower()
rows = []

for scenario, absolute_mde in mde_scenarios.items():
    target_rate = baseline_d30 + absolute_mde
    effect_size = abs(proportion_effectsize(target_rate, baseline_d30))
    sample_per_group = math.ceil(power_analysis.solve_power(
        effect_size=effect_size, alpha=alpha, power=power,
        ratio=1.0, alternative="two-sided"
    ))
    total_sample = sample_per_group * 2
    recruitment_days = math.ceil(total_sample / historical_avg_daily_eligible_users)
    rows.append({
        "scenario": scenario,
        "baseline_rate": baseline_d30,
        "target_rate": target_rate,
        "absolute_mde": absolute_mde,
        "relative_mde": absolute_mde / baseline_d30,
        "required_sample_per_group": sample_per_group,
        "total_required_sample": total_sample,
        "historical_avg_daily_eligible_users": historical_avg_daily_eligible_users,
        "recruitment_days": recruitment_days,
        "outcome_maturation_days": 30,
        "total_calendar_days": recruitment_days + 30
    })

sample_size_scenarios = pd.DataFrame(rows)
sample_size_scenarios.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
sample_size_scenarios.style.format({
    "baseline_rate": "{:.2%}", "target_rate": "{:.2%}",
    "absolute_mde": "{:.1%}", "relative_mde": "{:.2%}",
    "historical_avg_daily_eligible_users": "{:,.2f}"
})

,scenario,baseline_rate,target_rate,absolute_mde,relative_mde,required_sample_per_group,total_required_sample,historical_avg_daily_eligible_users,recruitment_days,outcome_maturation_days,total_calendar_days
0,Scenario A,2.37%,2.67%,0.3%,12.65%,42823,85646,"6,890.36",13,30,43
1,Scenario B,2.37%,2.87%,0.5%,21.09%,15990,31980,"6,890.36",5,30,35
2,Scenario C,2.37%,3.37%,1.0%,42.18%,4345,8690,"6,890.36",2,30,32


# 9. 实验招募周期

recruitment_days = total_required_sample / historical_average_daily_eligible_users，结果向上取整。总样本量已经包含50/50两组，因此不能再次除以2。历史日均流量只是planning proxy，正式上线前应使用近期真实流量重新估算。

# 10. 总实验日历周期

- Recruitment Period：收够目标样本所需时间；
- Outcome Maturation Window：最后一个入组用户达到D30所需约30天；
- total_calendar_days = recruitment_days + 30。

真实业务中还应额外考虑数据延迟、完整性检查和结果分析时间。

# 11. 实验上线后需要补充的数据

真实实验系统至少需要：

- msno
- experiment_id
- experiment_group
- assignment_time
- exposure_time
- treatment_version
- post-treatment D7/D30 outcome
- relevant revenue / subscription outcome

实验结束后才能进行SRM、Control/Treatment conversion comparison、uplift、confidence interval、p-value、statistical significance和rollout decision。当前历史KKBox数据缺少真实Control/Treatment exposure，不能完成这些评估。

# 12. 最终结论

本模块完成的是实验上线前设计，包括实验对象、Control/Treatment、用户级稳定50/50分流、Primary/Secondary/Guardrail metrics、历史baseline、三档MDE、样本量、招募周期和D30 observation window。

本模块没有真实post-treatment experiment data，因此不报告实验有效、D30提升、Treatment显著优于Control、uplift、p-value或置信区间。